# 07 – Features from Tennis Knowledge

**Objective:** turn ideas from following tennis (confidence, priorities, slumps) into features, and keep only those that improve predictions.

**Key results (walk-forward validation, 2015–2021)**
- **Only one idea clearly helps:** a player's past results at the same tournament lower log loss from 0.5856 to 0.5849.
- **The other seven ideas** change log loss by 0.0002 or less, so they make no meaningful difference.
- **Not testable:** partners or family in the stands, and coaches, because no dated data is available.

**Scope:** as in notebook 06, only matches up to 2021 are scored, so the test period stays untouched.

In [1]:
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from tennis_data import load_all_matches
from tennis_domain_features import DOMAIN_GROUPS, build_domain_features
from tennis_elo import run_elo
from tennis_features import FEATURE_GROUPS
from tennis_models import INTERACTIONS, add_interactions, logistic_model, score, walk_forward

ELO_SETTINGS = dict(k=28, inactivity=0.3, new_player_boost=1.0)
ELO_WEIGHT = 0.6

matches = load_all_matches("../data/raw")
ratings = run_elo(matches, **ELO_SETTINGS)
domain = build_domain_features(matches, ratings, ELO_WEIGHT)

conn = sqlite3.connect(Path("../data/processed/tennis.db"))
base = pd.read_sql_query("SELECT * FROM match_features WHERE tourney_level <> 'D' AND year <= 2021", conn)
data = add_interactions(base.merge(domain, on=["tourney_id", "match_num"], validate="one_to_one"))
validation = data["year"].between(2015, 2021)
print(f"{len(data):,} matches, {validation.sum():,} in validation")

59,320 matches, 17,126 in validation


## 1. Leakage test

Features for matches up to 2018 must be identical whether or not later years exist.

`d_top_before_slam` is left out of this comparison. It differs in 22 matches from events starting in late December 2018, because removing 2019 also removes the 2019 Australian Open from the calendar. Slam dates are published in advance, so this isn't leakage.

In [2]:
past = matches[matches["year"] <= 2018]
past_domain = build_domain_features(past, run_elo(past, **ELO_SETTINGS), ELO_WEIGHT)
columns = [c for group in DOMAIN_GROUPS.values() for c in group if c != "d_top_before_slam"]
full = domain.iloc[:len(past_domain)]
print("Same matches:", (full["match_num"].to_numpy() == past_domain["match_num"].to_numpy()).all())
print("Largest difference:", np.abs(full[columns].to_numpy() - past_domain[columns].to_numpy()).max())

Same matches: True
Largest difference: 0.0


## 2. Does each idea help?

Each idea is added **on its own** to the model from notebook 06. A positive improvement means lower log loss.

In [3]:
ORDER = ["Elo", "Ranking", "Serve and return", "Form", "Fatigue and fitness", "Experience",
         "Style and age", "Head to head", "Draw", "Home"]
BASE_COLUMNS = [c for group in ORDER for c in FEATURE_GROUPS[group]] + list(INTERACTIONS)

base_score = score(walk_forward(data, BASE_COLUMNS, logistic_model)[validation])
rows = [{"idea": "Model from notebook 06", **base_score, "improvement": 0.0}]
for idea, extra in DOMAIN_GROUPS.items():
    result = score(walk_forward(data, BASE_COLUMNS + extra, logistic_model)[validation])
    rows.append({"idea": idea, **result, "improvement": round(base_score["log_loss"] - result["log_loss"], 4)})

pd.DataFrame(rows)

,idea,accuracy_pct,log_loss,improvement
0,Model from notebook 06,68.46,0.5856,0.0000
1,Season confidence,68.39,0.5855,0.0001
2,Grand Slam priority,68.43,0.5858,-0.0002
3,Top player in small event,68.45,0.5856,0.0000
4,After a big title,68.46,0.5856,0.0000
5,Tournament history,68.38,0.5849,0.0007
6,End of season,68.45,0.5857,-0.0001
7,Comeback,68.42,0.5857,-0.0001
8,Slump,68.46,0.5855,0.0001


## 3. All the ideas together

In [4]:
all_ideas = [c for group in DOMAIN_GROUPS.values() for c in group]
pd.DataFrame([
    {"model": "Model from notebook 06", **base_score},
    {"model": "With all ideas", **score(walk_forward(data, BASE_COLUMNS + all_ideas, logistic_model)[validation])},
])

,model,accuracy_pct,log_loss
0,Model from notebook 06,68.46,0.5856
1,With all ideas,68.42,0.5849


## 4. Conclusions

| Idea | Log loss improvement | Decision |
|---|---|---|
| Tournament history | 0.0007 | Added to the final model |
| Season confidence | 0.0001 | Drop: Elo and recent form already capture it |
| Slump | 0.0001 | Drop |
| Top player in small event | 0.0000 | Drop |
| After a big title | 0.0000 | Drop |
| End of season | -0.0001 | Drop |
| Comeback | -0.0001 | Drop |
| Grand Slam priority | -0.0002 | Drop |

**Limits**
- **These features are rough proxies.** A weak result doesn't prove that an idea is wrong, only that this version of it adds nothing beyond the existing features.
- **Even tournament history is a small gain,** chosen on the validation years, so it could shrink on new data.
- **Accuracy didn't improve:** adding all the ideas moves it from 68.46% to 68.42%.

## 5. The final model

The final model is the notebook 06 logistic regression plus the three tournament history features. They are saved to a new table, `final_features`, ready for the final test. Years from 2022 are saved too, but they aren't scored here.

In [5]:
HISTORY = DOMAIN_GROUPS["Tournament history"]
FINAL_COLUMNS = BASE_COLUMNS + HISTORY

everything = pd.read_sql_query("SELECT * FROM match_features", conn)
final_features = everything.merge(domain[["tourney_id", "match_num"] + HISTORY], on=["tourney_id", "match_num"], validate="one_to_one")
final_features.to_sql("final_features", conn, if_exists="replace", index=False)
conn.commit()

print(f"Saved {len(final_features):,} matches. The final model uses {len(FINAL_COLUMNS)} columns.")
score(walk_forward(data, FINAL_COLUMNS, logistic_model)[validation])

Saved 78,876 matches. The final model uses 60 columns.


{'accuracy_pct': 68.38, 'log_loss': 0.5849}